# Basic implementation for the CKKS algorithm using Pyphel

## Author: José Ángel de Bustos Pérez
## License: [GNU GENERAL PUBLIC LICENSE v3](https://www.gnu.org/licenses/gpl-3.0.html)

This notebook uses [Pyphel](https://pyfhel.readthedocs.io/) to implement the CKKS homomorphic encryption algorithm.

CKKS (Cheon–Kim–Kim–Song, 2017) is a homomorphic encryption scheme based on RLWE (Ring Learning With Errors), like BFV, but designed to work with real or complex numbers in an approximate way. That means that the operations will add some error.

Key differences compared to BFV:

* Data: float/double, not integers.
* Results: approximate (there is rounding error).
* Rescaling: after each multiplication, rescaling is required.
* Levels: each rescale consumes a level of the ciphertext.
* Slots: n/2 instead of n (slots are complex conjugates).

It is the standard scheme for machine learning on encrypted data, statistics, signal processing, etc.

The first step is to set the parameters and generate the keys:

* **n**, polynomial degree (power of 2). In CKKS, the number of slots is n/2.
* **scale**, scaling factor. CKKS encodes floats by multiplying them by this factor to convert them into large integers. A larger scale gives more decimal precision but adds more noise. Typical scale value used: 2^30.
* **qi_sizes**, sizes (in bits) of the prime moduli that form the modulus chain. CKKS uses modulus switching: each rescale after a multiplication removes one prime from the chain. The number of intermediate primes equals the multiplicative depth you can support.
* **sec**, security level in bits.

In [1]:
from Pyfhel import Pyfhel
import numpy as np

HE = Pyfhel()

ckks_params = {
    'scheme': 'CKKS',
    'n': 2**14,                # 16384 coefficients => 8192 slots for batching
    'scale': 2**30,            # approximate precision: ~9 decimal places
    # qi_sizes gives a chain of 7 primes. The first and last are “large” (60 bits) for technical reasons; 
    # the ones in the middle are the size of the scale (30 bits). ⇒ Approximately 5 levels of multiplication.
    'qi_sizes': [60, 30, 30, 30, 30, 30, 60],
    'sec': 128,
}

HE.contextGen(**ckks_params)
HE.keyGen()           # Key creation
HE.rotateKeyGen()     # Required for rotations (we will use it in batching)
HE.relinKeyGen()      # Required to reduce size after multiplication

print(f"Scheme: {HE.scheme}")
print(f"n (Polynomial degree): {HE.n}")
print(f"Available Slots (n/2): {HE.n // 2}")
print(f"Scale: 2^{int(np.log2(HE.scale))}")
print(f"qi modulus chain: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

Scheme: Scheme_t.ckks
n (Polynomial degree): 16384
Available Slots (n/2): 8192
Scale: 2^30
qi modulus chain: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5


Unlike BFV, in CKKS the **noise budget** is not measured in the same way. What matters is the LEVEL of the ciphertext (how many primes remain in the modulus chain). Each rescale consumes one level. When you reach the last level, you can no longer perform multiplications.

## Basic operations

We will perform basic operations such as addition and multiplications to check how noise is increasing. We will start with addition:

In [2]:
import random

# Randon number generation
a = random.random()
b = random.random()

expected_value = a + b

# Encrypting data. CKKS always works with vectors. To encrypt a scalar, we place it
# in an array (the remaining slots are filled with zeros).
fhe_a = HE.encryptFrac(np.array([a], dtype=np.float64))
fhe_b = HE.encryptFrac(np.array([b], dtype=np.float64))

# Homomorphic operation (addition)
fhe_suma = fhe_a + fhe_b

# Getting the operation value, decrypting
suma = HE.decryptFrac(fhe_suma)[0]

# Error
error = abs(expected_value - suma)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic addition: {suma}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.8734162648318591, b = 0.5600002617345973
Expected value: 1.4334165265664565
Homomorphic addition: 1.4334178993913267
Error: 1.37e-06


We are going to proceed with multiplication.

In [3]:
expected_value = a * b

# Homomorphic multiplication
fhe_mult = fhe_a * fhe_b
# Relinearisation
~fhe_mult

# Rescalation. After each multiplication, we must manually rescale. CKKS multiplies two numbers scaled by 
# S, and the result ends up scaled by S^2. The rescale operation divides by S and removes one prime from the 
# chain to keep the original scaling factor S
HE.rescale_to_next(fhe_mult)

# Getting the operation value, decrypting
mult = HE.decryptFrac(fhe_mult)[0]

# Error
error = abs(expected_value - mult)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic multiplication: {mult}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.8734162648318591, b = 0.5600002617345973
Expected value: 0.4891133369090954
Homomorphic multiplication: 0.4891175909387754
Error: 4.25e-06


## Performing homomorphic encryption with multiple operations

We are going to show how to perform more complex operations with homomorphic encryption. Let's assume we want to perform homomorphic encryption to:

$$f(x,y) = (x+y)^2 - (x-y)^2$$

In [4]:
# Random number generation
x = random.random()
y = random.random()

expected_value = (x + y)**2 - (x-y)**2

# Encrypting data
fhe_x = HE.encryptFrac(np.array([x], dtype=np.float64))
fhe_y = HE.encryptFrac(np.array([y], dtype=np.float64))

# (x + y)
fhe_addition_xy = fhe_x + fhe_y

# (x + y)^2
fhe_square_add = fhe_addition_xy * fhe_addition_xy
# Relinearisation
~fhe_square_add
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_add)

# (x - y)
fhe_substraction_xy = fhe_x - fhe_y

# (x - y)^2
fhe_square_substraction = fhe_substraction_xy * fhe_substraction_xy
# Relinearisation
~fhe_square_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_substraction)

# Final. Before doing the substraction both ciphertext must have the same
# level, in both we have performed one rescale operation so they are at the
# same level.
fhe_final = fhe_square_add - fhe_square_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**2")
print(f"Expected value: {expected_value}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.4481619246690438 + 0.9626322040826019)**2 - (0.4481619246690438-0.9626322040826019)**2
Expected value: 1.7256604053202504
Homomorphic value: 1.7256583265231864
Error: 2.08e-06


Added noise is mostly the same that the noise added by the homomorphic multiplication. Now, we will perform the following:

$$f(x,y) = (x+y)^2 - (x-y)^3$$

In this example $(x+y)^2$ has a multiplicativity depth of 1 but $(x-y)^3$ has a multiplicativity depth of 2. That means that we will need to align levels (working on the same scale) to operate with them.

In [5]:
expected_value_third = (x + y)**2 - (x - y)**3

# Aligning data on the same scale. fhe_substraction_xy is in level 1
# but fhe_square_substraction is in level 2
fhe_substraction_xy_aligned = fhe_substraction_xy.copy()
HE.mod_switch_to_next(fhe_substraction_xy_aligned)

# (x - y)^3
fhe_third_substraction = fhe_square_substraction * fhe_substraction_xy_aligned
# Relinearisation
~fhe_third_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_third_substraction)

# (x + y)^2 is in level 1 (one rescaling) but (x - y)^3 is in level 2
# before operating them, we need to have both of them on the  same level
fhe_square_add_aligned = fhe_square_add.copy()
HE.mod_switch_to_next(fhe_square_add_aligned)

# Final
fhe_final = fhe_square_add_aligned - fhe_third_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value_third - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**3")
print(f"Expected value: {expected_value_third}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.4481619246690438 + 0.9626322040826019)**2 - (0.4481619246690438-0.9626322040826019)**3
Expected value: 2.126509896676881
Homomorphic value: 2.1267383219406453
Error: 2.28e-04


## Batching operation

Batching allows us to perform the same homomorphic operation to several data at the same time, parallelism.

If we are familiar with processor architectures we will know what **SIMD** (**S**imple **I**nstruction **M**ultiple **D**ata) is. One single operation using only one clock cicle operate on multiple data. Batching is the same but used in homomorphic encryption and it is used to speed up operations.

In [6]:
# Randon number generation
size = 10 # number of floats to operate at the same time
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypting data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Item by item homomorphic addition
fhe_add_vectorial = fhe_v1 + fhe_v2

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_add_vectorial)[:size]

# Error
expected_value = v1 + v2
error = np.abs(final - expected_value)

print(f"v1: {v1}")
print(f"v2: {v2}")
print(f"Homomorphic v1 + v2:  {final}")
print(f"Expected value: {expected_value}")
print(f"Error: {error}")

v1: [ 35.75972991  66.68896621 -36.70555792  -1.67841515  29.46138676
  85.54655296  -8.11154399  61.17539204  11.35365933  83.99717032]
v2: [-52.7463701   -1.20571368 -58.37749234 -46.55361161 -63.34808723
  76.41688167  45.60264234 -68.41979202 -90.04665375  92.51011634]
Homomorphic v1 + v2:  [-16.98663877  65.48325129 -95.08304948 -48.23202956 -33.88670288
 161.96343038  37.49109871  -7.24439441 -78.69299811 176.50728888]
Expected value: [-16.98664019  65.48325253 -95.08305026 -48.23202676 -33.88670047
 161.96343463  37.49109836  -7.24439999 -78.69299442 176.50728667]
Error: [1.42220031e-06 1.24085506e-06 7.83355460e-07 2.79521014e-06
 2.40756462e-06 4.24512388e-06 3.59238463e-07 5.57154294e-06
 3.69457958e-06 2.20868043e-06]


Let's see what happens with batching multiplication:

In [7]:
# Item by item homomorphic multiplication
fhe_multiplication_vectorial = fhe_v1 * fhe_v2
# Relinearisation
~fhe_multiplication_vectorial

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_multiplication_vectorial)[:size]

# Error
expected_value = v1 * v2
error = np.abs(final - expected_value)

print(f"\nHomomorphic v1 * v2  = {final}")
print(f"Expected value = {v1 * v2}")
print(f"Error: {error}")


Homomorphic v1 * v2  = [-1886.19597536   -80.40798123  2142.77838284    78.13640042
 -1866.32256229  6537.20046151  -369.90781637 -4185.6077151
 -1022.35889389  7770.58818944]
Expected value = [-1886.19594847   -80.40779863  2142.77842633    78.13628705
 -1866.32249798  6537.2008144   -369.9078393  -4185.60759992
 -1022.35903092  7770.58799916]
Error: [2.68881890e-05 1.82603032e-04 4.34944768e-05 1.13366010e-04
 6.43099313e-05 3.52896231e-04 2.29283643e-05 1.15179139e-04
 1.37036475e-04 1.90283768e-04]


## Dot product or scalar product

Dot product or scalar product is a fundamental operation used in multiple algorithms such as linear regresion, support vector machines (SVM) or neural network algorithms. For this reason been able to operate it using homomorphic encription will ease using such algoritms with homomorphic encryption.

In [8]:
# Random sample
size = 10 # random sample size
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypt data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Cdot operation with encrypted data
fhe_cdot = fhe_v1 * fhe_v2
# Relinearisation
~fhe_cdot

# vector with product, component to component
value_cdot = HE.decryptFrac(fhe_cdot)[:size]

# Error
expected_value = float(np.dot(v1, v2))
error = abs(expected_value - np.sum(value_cdot))

print(f"v1: {v1}")
print(f"v2: {v2}")
# cdot product is the sum for all the values in value_cdot
print(f"Homomorphic v1 * v2: {np.sum(value_cdot)}")
print(f"Expected value: {expected_value}")
print(f"Error: {error:.2e}")

v1: [-43.72309844  20.35239636 -10.566623    69.73268705  33.64658174
  57.42237231  31.15745582 -62.96673853  46.87562564  91.43962242]
v2: [-94.17232464  88.15046455  89.75166231  82.12875221 -88.93177432
 -54.27995538 -86.84974933 -93.09658306 -57.55125403 -87.10585993]
Homomorphic v1 * v2: -2925.5756185778655
Expected value: -2925.5754158740247
Error: 2.03e-04


## How many operations can be done before data is corrupted?

We have seen that the multiplication operation adds noise to the data. For this reason is important to know how many operations can be done without corrupting data. This number  of operations is the operational limit which indicates the maximum number of operations that can be performed by the algorithm.

In CKKS the limit is the prime number chain defined in **qi_sizes = [60, 30, 30, 30, 30, 30, 60]**. With each rescaling operation one prime number is used, in this case we have five multiplication levels

In [9]:
from random import randrange

data = randrange(10)
fhe_data = HE.encryptFrac(np.array([data], dtype=np.float64))

print(f"Encrypting data = {data}")
print(f"Initial qi_sizes: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

# Start multiplying encrypted data
for i in range(1, 10):
    try:
        fhe_data = fhe_data * fhe_data   
        # Relinearisation
        ~fhe_data
        # Rescaling
        HE.rescale_to_next(fhe_data)
        expected_value = data ** (2 ** i)
        fhe_value = HE.decryptFrac(fhe_data)[0]
        error = abs(expected_value - fhe_value)
        print(f"Iteration {i}: data^{2**i:<5} | "
              f"Expected value={expected_value:<20.6f} "
              f"Encrypted value={fhe_value:<20.6f} | "
              f"Error={error:.2e}")
    except Exception as e:
        print(f"\nIteration {i}: FALLO - no more multiplication levels available.")
        print(f"  Excepción: {type(e).__name__}: {e}")
        break

Encrypting data = 2
Initial qi_sizes: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5
Iteration 1: data^2     | Expected value=4.000000             Encrypted value=4.000021             | Error=2.09e-05
Iteration 2: data^4     | Expected value=16.000000            Encrypted value=16.000165            | Error=1.65e-04
Iteration 3: data^8     | Expected value=256.000000           Encrypted value=256.005286           | Error=5.29e-03
Iteration 4: data^16    | Expected value=65536.000000         Encrypted value=65538.706327         | Error=2.71e+00


Iteration 5: data^32    | Expected value=4294967296.000000    Encrypted value=4295322026.706947    | Error=3.55e+05

Iteration 6: FALLO - no more multiplication levels available.
  Excepción: ValueError: scale out of bounds
